In [1]:
import sys
import pandas as pd
from itertools import combinations
from collections import Counter

# Import our custom data loader
sys.path.append('../src')
from data_prep import load_and_split_data

# 1. Load the data
train_data, test_data, orders = load_and_split_data()

# 2. Memory Optimization: Filter for the top 2000 most popular products
print("Filtering data for top products to optimize memory...")
top_2000_products = train_data['product_id'].value_counts().head(2000).index
filtered_carts = train_data[train_data['product_id'].isin(top_2000_products)]

# 3. Group products by their order_id
print("Grouping items by cart...")
# This creates a pandas Series where each row is a list of product IDs in a single order
carts = filtered_carts.groupby('order_id')['product_id'].apply(list)

# 4. Count the Pairs
print("Counting co-occurring pairs (this may take a minute)...")
pair_counts = Counter()

for items in carts:
    # We only care about carts with at least 2 items
    if len(items) > 1:
        # Sort items so (A, B) is counted the same as (B, A)
        items.sort()
        # Find all 2-item combinations in this cart and add to our counter
        pair_counts.update(combinations(items, 2))

# Display the top 10 most common pairs
print("\nTop 10 Most Frequently Bought Together Pairs:")
for pair, count in pair_counts.most_common(10):
    print(f"Product IDs {pair}: {count:,} times")

Loading datasets...
Merging data to create time-based Train/Test splits...
Split complete!
Filtering data for top products to optimize memory...
Grouping items by cart...
Counting co-occurring pairs (this may take a minute)...

Top 10 Most Frequently Bought Together Pairs:
Product IDs (13176, 47209): 62,341 times
Product IDs (13176, 21137): 61,628 times
Product IDs (21137, 24852): 56,156 times
Product IDs (24852, 47766): 53,395 times
Product IDs (21903, 24852): 51,395 times
Product IDs (13176, 21903): 50,372 times
Product IDs (16797, 24852): 41,232 times
Product IDs (24852, 47626): 40,880 times
Product IDs (21137, 47209): 40,794 times
Product IDs (13176, 27966): 40,503 times


In [3]:
def get_cart_suggestions_for_list(cart_items, rules_dataframe, top_n=5):
    """
    Accepts a list of items currently in the cart and returns top 5 complementary add-ons.
    """
    all_suggestions = pd.DataFrame()
    
    for product_name in cart_items:
        matches_A = rules_dataframe[rules_dataframe['item_A_name'].str.lower() == product_name.lower()][['item_B_name', 'lift']].rename(columns={'item_B_name': 'Suggestion'})
        matches_B = rules_dataframe[rules_dataframe['item_B_name'].str.lower() == product_name.lower()][['item_A_name', 'lift']].rename(columns={'item_A_name': 'Suggestion'})
        all_suggestions = pd.concat([all_suggestions, matches_A, matches_B])
    
    if all_suggestions.empty:
        return "No strong suggestions found for this cart."
        
    # Remove items that are already in the cart!
    cart_lower = [item.lower() for item in cart_items]
    filtered_suggestions = all_suggestions[~all_suggestions['Suggestion'].str.lower().isin(cart_lower)]
    
    # Combine the lift scores if multiple cart items suggest the same product
    final_ranked = (filtered_suggestions
                    .groupby('Suggestion')['lift']
                    .sum()
                    .sort_values(ascending=False)
                    .head(top_n)
                    .reset_index())
    
    return final_ranked

# Test the upgraded list function!
current_cart = ['organic cilantro', 'limes']
print(f"\nIf a user has {current_cart} in their cart, we suggest:")
display(get_cart_suggestions_for_list(current_cart, rules_df))


If a user has ['organic cilantro', 'limes'] in their cart, we suggest:


,Suggestion,lift
0,Organic Jalapeno Pepper,15.015213
1,Lemon Grass,13.968152
2,Poblano Pepper,12.954449
3,Serrano Chile Peppers,12.186626
4,Jalapeno Peppers,11.907741
